# Rolling Each-Ticker Training with Aggregated Evaluation

This notebook trains one model per ticker for each rolling two-year training window, predicts only that ticker's next test year, and then aggregates the out-of-sample results across the common ticker universe.

Primary outputs:
- ticker × fold within-ticker Spearman;
- ticker-level mean/median performance across folds;
- universe-level mean and median within-ticker Spearman;
- positive ticker rate;
- ticker-level distribution summaries;
- fold-level stability;
- row-level predictions for later paired comparison with a pooled model;
- primary paired pooled-versus-single comparison with one pair per ticker, using only common test folds;\n- secondary ticker × fold paired analysis for temporal stability.

The pooled notebook should export row-level predictions with matching ticker, test_year, target_name, model and IndexReference fields.

The pooled comparison reads the supplied pooled parquet, filters to `rolling_2y`, `rl_long_current_pnl`, `full_feature_B`, and the same models, then matches rows exactly on `IndexReference`, ticker, test year, target, feature set and model before calculating ticker-level paired Spearman results.

All headline averages, medians, positive rates, paired tests, variance comparisons and fold-level stability summaries use complete test years 2022–2025. The partial 2026 fold is exported separately and is not included in the main summaries.

Regression models: ElasticNet, RandomForest and LightGBM. ElasticNet uses median imputation followed by StandardScaler fitted inside each training fold, with alpha=0.001, l1_ratio=0.5 and max_iter=5000, matching the pooled notebook.

In [1]:
# ============================================================
# 1. Imports and global settings
# ============================================================
import os
import json
import gzip
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier
from scipy import stats

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score, explained_variance_score,
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_JOBS = -1

# Canonical column names.
INDEX_COL = 'IndexReference'
DATE_COL = 'attr__timestamp'
TICKER_COL = 'attr__ticker'
SIC2_COL = 'attr__sic2'
YEAR_COL = 'year'
SPLIT_COL = 'split'
TRUE_COL = 'y_true'
PRED_COL = 'y_pred'
SCORE_COL = 'prediction_score'
SIGNAL_SCORE_COL = 'signal_score'
CONFIDENCE_COL = 'confidence'
RL_TRAINABLE_COL = 'label__rl.trainable'

PROJECT_ROOT = Path.cwd()
OUTPUT_BASE_DIR = PROJECT_ROOT / 'rolling_each_ticker_aggregated_outputs'
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT.resolve())
print('Output base directory:', OUTPUT_BASE_DIR.resolve())


Project root: C:\Users\user\Downloads\universe_100_recent_post_normalisation_single_ticker_0721
Output base directory: C:\Users\user\Downloads\universe_100_recent_post_normalisation_single_ticker_0721\rolling_each_ticker_aggregated_outputs


In [2]:
# ============================================================
# 2. User configuration
# ============================================================
UNIVERSE = 'universe_100'
TIME_HORIZON = 'recent'
NORMALISATION_STATUS = 'post_normalisation'

DATASET_CONFIG = {
    'train_path': PROJECT_ROOT / f'{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_train.jsonl',
    'valid_path': PROJECT_ROOT / f'{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_validation.jsonl',
    'test_path':  PROJECT_ROOT / f'{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_test.jsonl',
    'project_b_feature_file': PROJECT_ROOT / 'feature_project_b.txt',
}

# None means: automatically use every ticker available after data loading.
# Alternatively provide an explicit common universe, e.g. ['AAPL', 'MSFT', ...].
TICKERS = None

# Main dissertation target. Add other targets only when required.
TARGET_NAMES = ['rl_long_current_pnl']
FEATURE_SET_NAME = 'full_feature_B'

# Fair pooled-versus-single comparison: same two-year rolling windows and test years.
ROLLING_TRAIN_YEARS = 2
MAIN_TEST_YEARS = [2022, 2023, 2024, 2025]
PARTIAL_TEST_YEARS = [2026]
N_TEST_FOLDS = 5
TEST_YEARS = None  # Or explicitly set common years, e.g. [2022, 2023, 2024, 2025, 2026].

MIN_TRAIN_ROWS = 250
MIN_TEST_ROWS = 40
TOP_BOTTOM_PCT = 0.10

# Keep the same model names/settings as the pooled notebook for a controlled comparison.
REGRESSION_MODELS = ['ElasticNet', 'RandomForest', 'LightGBM']
BINARY_MODELS = ['Logistic', 'RandomForest', 'LightGBM']
MULTICLASS_MODELS = ['LogisticMultinomial', 'RandomForest', 'LightGBM']

# Optional pooled row-level prediction file for the paired comparison cell.
# Leave as None until the pooled notebook has exported its predictions.
POOLED_PREDICTIONS_PATH = Path(r'/mnt/data/walk_forward_predictions_20260721_030422_pi_hindsight_entry_long_pi_hindsight_entry_original_rl_long_.parquet')
# Example:
# POOLED_PREDICTIONS_PATH = PROJECT_ROOT / 'pooled_rolling_row_level_predictions.parquet'

for k, p in DATASET_CONFIG.items():
    print(k, '->', p, '| exists:', Path(p).exists())

train_path -> c:\Users\user\Downloads\universe_100_recent_post_normalisation_single_ticker_0721\universe_100_recent_post_normalisation_train.jsonl | exists: True
valid_path -> c:\Users\user\Downloads\universe_100_recent_post_normalisation_single_ticker_0721\universe_100_recent_post_normalisation_validation.jsonl | exists: True
test_path -> c:\Users\user\Downloads\universe_100_recent_post_normalisation_single_ticker_0721\universe_100_recent_post_normalisation_test.jsonl | exists: True
project_b_feature_file -> c:\Users\user\Downloads\universe_100_recent_post_normalisation_single_ticker_0721\feature_project_b.txt | exists: False


In [3]:
# ============================================================
# 3. Data loading utilities
# ============================================================
def _open_text(path):
    path = Path(path)
    if path.suffix.lower() == '.gz':
        return gzip.open(path, 'rt', encoding='utf-8')
    return path.open('r', encoding='utf-8')


def flatten_record(raw_record):
    """Flatten Adaptive Swarm JSONL rows into attr__/feature__/label__ columns."""
    if raw_record.get('section') == 'header':
        return None
    if raw_record.get('section') == 'data' and isinstance(raw_record.get('data'), dict):
        rec = raw_record['data']
    else:
        rec = raw_record
    row = {}
    if INDEX_COL in rec:
        row[INDEX_COL] = rec.get(INDEX_COL)
    for family, prefix in [('Attributes', 'attr__'), ('attributes', 'attr__'), ('Features', 'feature__'), ('features', 'feature__'), ('Labels', 'label__'), ('labels', 'label__')]:
        obj = rec.get(family)
        if isinstance(obj, dict):
            for k, v in obj.items():
                row[f'{prefix}{k}'] = v
    # Keep any already-flat columns too.
    for k, v in rec.items():
        if k not in ['Attributes', 'attributes', 'Features', 'features', 'Labels', 'labels'] and k not in row:
            row[k] = v
    return row


def load_jsonl(path):
    rows = []
    with _open_text(path) as f:
        for line in f:
            if not line.strip():
                continue
            row = flatten_record(json.loads(line))
            if row is not None:
                rows.append(row)
    return pd.DataFrame(rows)


def load_any_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    suffixes = ''.join(path.suffixes).lower()
    if suffixes.endswith('.jsonl') or suffixes.endswith('.jsonl.gz'):
        return load_jsonl(path)
    if suffixes.endswith('.parquet'):
        return pd.read_parquet(path)
    if suffixes.endswith('.csv'):
        return pd.read_csv(path)
    raise ValueError(f'Unsupported file type: {path}')


def load_dataset_from_config(config):
    frames = {}
    for split, key in [('train', 'train_path'), ('valid', 'valid_path'), ('test', 'test_path')]:
        path = Path(config[key])
        if path.exists():
            df = load_any_table(path)
            df[SPLIT_COL] = split
            frames[split] = df
            print(split, df.shape, path.name)
        else:
            print(f'WARNING: {split} path not found:', path)
    if not frames:
        raise FileNotFoundError('No train/valid/test files were found. Update DATASET_CONFIG paths first.')
    all_data = pd.concat(frames.values(), ignore_index=True, sort=False)
    return frames, all_data


In [4]:
# ============================================================
# 4. Target construction
# ============================================================
def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def make_pi_hindsight_entry_long_6bins(series):
    s = pd.to_numeric(series, errors='coerce')
    out = pd.Series(np.nan, index=s.index)
    out[s == 0] = 0
    out[(s > 0) & (s < 0.1)] = 1
    out[(s >= 0.1) & (s < 0.2)] = 2
    out[(s >= 0.2) & (s < 0.3)] = 3
    out[(s >= 0.3) & (s < 0.4)] = 4
    out[s >= 0.4] = 5
    return out.astype('Int64')


def add_derived_targets(df):
    df = df.copy()

    pi_col = first_existing_column(df, [
        'label__pi_hindsight_entry_long', 'label__pi_long_entry', 'pi_hindsight_entry_long',
        'target__pi_hindsight_entry_long'
    ])
    if pi_col is not None:
        df['target__pi_hindsight_entry_long'] = pd.to_numeric(df[pi_col], errors='coerce')
        df['target__pi_hindsight_entry_positive'] = (df['target__pi_hindsight_entry_long'] > 0).astype('Int64')
        df['target__pi_hindsight_entry_original'] = (df['target__pi_hindsight_entry_long'] >= 0.4).astype('Int64')
        df['target__pi_hindsight_entry_6bins'] = make_pi_hindsight_entry_long_6bins(df['target__pi_hindsight_entry_long'])

    col = first_existing_column(df, ['label__rl.expert_action', 'label__rl_expert_action', 'target__rl_expert_action'])
    if col is not None:
        df['target__rl_expert_action'] = df[col]

    col = first_existing_column(df, ['label__rl.long.action_label', 'label__rl_long_action_binary', 'target__rl_long_action_binary'])
    if col is not None:
        df['target__rl_long_action_binary'] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    col = first_existing_column(df, ['label__rl.long_is_best', 'label__rl_long_is_best', 'target__rl_long_is_best'])
    if col is not None:
        df['target__rl_long_is_best'] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    col = first_existing_column(df, ['label__rl.long.action_quality', 'label__rl_long_action_quality', 'target__rl_long_action_quality'])
    if col is not None:
        df['target__rl_long_action_quality'] = pd.to_numeric(df[col], errors='coerce')

    col = first_existing_column(df, ['label__rl.long.current_pnl', 'label__rl_long_current_pnl', 'label__rl.long_current_pnl', 'label__current_pnl', 'target__rl_long_current_pnl'])
    if col is not None:
        df['target__rl_long_current_pnl'] = pd.to_numeric(df[col], errors='coerce')

    return df


TARGET_CONFIGS = {
    'pi_hindsight_entry_long': {'column': 'target__pi_hindsight_entry_long', 'task': 'regression'},
    'pi_hindsight_entry_positive': {'column': 'target__pi_hindsight_entry_positive', 'task': 'binary'},
    'pi_hindsight_entry_original': {'column': 'target__pi_hindsight_entry_original', 'task': 'binary'},
    'pi_hindsight_entry_6bins': {'column': 'target__pi_hindsight_entry_6bins', 'task': 'multiclass'},
    'rl_expert_action': {'column': 'target__rl_expert_action', 'task': 'multiclass'},
    'rl_long_action_binary': {'column': 'target__rl_long_action_binary', 'task': 'binary'},
    'rl_long_is_best': {'column': 'target__rl_long_is_best', 'task': 'binary'},
    'rl_long_action_quality': {'column': 'target__rl_long_action_quality', 'task': 'regression'},
    'rl_long_current_pnl': {'column': 'target__rl_long_current_pnl', 'task': 'regression'},
}


In [5]:
# ============================================================
# 5. Feature selection
# ============================================================
def read_project_b_features(path):
    path = Path(path)
    if not path.exists():
        return []
    feats = []
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        feats.append(line)
    return feats


def build_feature_sets(df, config):
    numeric_feature_cols = [
        c for c in df.columns
        if c.startswith('feature__') and pd.api.types.is_numeric_dtype(pd.to_numeric(df[c], errors='coerce'))
    ]
    project_b_raw = read_project_b_features(config.get('project_b_feature_file', ''))
    # Support feature names either with or without feature__ prefix.
    project_b = []
    for f in project_b_raw:
        candidates = [f, f'feature__{f}' if not f.startswith('feature__') else f]
        for c in candidates:
            if c in df.columns:
                project_b.append(c)
                break
    project_b = sorted(set(project_b))
    if not project_b:
        project_b = numeric_feature_cols
    return {
        'combined_project_b': project_b,
        'all_numeric_features': numeric_feature_cols,
    }


In [6]:
# ============================================================
# 6. Models and metrics
# ============================================================
def make_numeric_preprocessor(scale=False):
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    return Pipeline(steps)


def make_regression_models():
    models = {
        'DummyMean': DummyRegressor(strategy='mean'),
        'ElasticNet': make_pipeline(make_numeric_preprocessor(True), ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestRegressor(n_estimators=200, min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMRegressor
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        print('LightGBM not available; using HistGradientBoosting fallback only.')
    return models


def make_binary_models():
    models = {
        'DummyMostFrequent': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
        'Logistic': make_pipeline(make_numeric_preprocessor(True), LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestClassifier(n_estimators=200, min_samples_leaf=10, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        print('LightGBM not available; using HistGradientBoosting fallback only.')
    return models


def make_multiclass_models():
    models = {
        'DummyMostFrequent': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
        'LogisticMultinomial': make_pipeline(make_numeric_preprocessor(True), LogisticRegression(class_weight='balanced', max_iter=3000, random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestClassifier(n_estimators=200, min_samples_leaf=10, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        print('LightGBM not available; using HistGradientBoosting fallback only.')
    return models


def get_models_for_task(task):
    if task == 'regression':
        return make_regression_models()
    if task == 'binary':
        return make_binary_models()
    if task == 'multiclass':
        return make_multiclass_models()
    raise ValueError(task)


def safe_spearman(y_true, y_score):
    y_true = pd.Series(y_true)
    y_score = pd.Series(y_score)
    valid = y_true.notna() & y_score.notna()
    if valid.sum() < 3 or y_true.loc[valid].nunique() < 2 or y_score.loc[valid].nunique() < 2:
        return np.nan
    return y_true.loc[valid].corr(y_score.loc[valid], method='spearman')


def safe_pearson(y_true, y_score):
    y_true = pd.Series(y_true)
    y_score = pd.Series(y_score)
    valid = y_true.notna() & y_score.notna()
    if valid.sum() < 3 or y_true.loc[valid].nunique() < 2 or y_score.loc[valid].nunique() < 2:
        return np.nan
    return y_true.loc[valid].corr(y_score.loc[valid], method='pearson')


def regression_metrics(y_true, y_pred):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred).astype(float)
    valid = y_true.notna() & y_pred.notna()
    if valid.sum() == 0:
        return {}
    yt = y_true.loc[valid]
    yp = y_pred.loc[valid]
    return {
        'n': int(valid.sum()),
        'mae': mean_absolute_error(yt, yp),
        'rmse': np.sqrt(mean_squared_error(yt, yp)),
        'r2': r2_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'explained_variance': explained_variance_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'pearson': safe_pearson(yt, yp),
        'spearman': safe_spearman(yt, yp),
        'directional_accuracy': (np.sign(yt) == np.sign(yp)).mean() if yt.nunique() > 1 and yp.nunique() > 1 else np.nan,
    }


def binary_metrics(y_true, y_pred, y_score=None):
    yt = pd.Series(y_true).dropna().astype(int)
    yp = pd.Series(y_pred).loc[yt.index].astype(int)
    out = {
        'n': int(len(yt)),
        'accuracy': accuracy_score(yt, yp),
        'balanced_accuracy': balanced_accuracy_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'precision': precision_score(yt, yp, zero_division=0),
        'recall': recall_score(yt, yp, zero_division=0),
        'f1': f1_score(yt, yp, zero_division=0),
    }
    if y_score is not None:
        ys = pd.Series(y_score).loc[yt.index]
        if yt.nunique() > 1 and ys.nunique() > 1:
            out['roc_auc'] = roc_auc_score(yt, ys)
            out['pr_auc'] = average_precision_score(yt, ys)
        else:
            out['roc_auc'] = np.nan
            out['pr_auc'] = np.nan
    return out


def multiclass_metrics(y_true, y_pred):
    # Convert to string to avoid sklearn errors with pandas nullable integer / mixed object labels.
    yt = pd.Series(y_true).dropna()
    yp = pd.Series(y_pred).loc[yt.index]
    yt_eval = yt.astype(str)
    yp_eval = yp.astype(str)
    return {
        'n': int(len(yt_eval)),
        'accuracy': accuracy_score(yt_eval, yp_eval),
        'balanced_accuracy': balanced_accuracy_score(yt_eval, yp_eval) if yt_eval.nunique() > 1 else np.nan,
        'macro_f1': f1_score(yt_eval, yp_eval, average='macro', zero_division=0),
        'weighted_f1': f1_score(yt_eval, yp_eval, average='weighted', zero_division=0),
        'n_classes_true': yt_eval.nunique(),
        'n_classes_pred': yp_eval.nunique(),
    }


In [7]:
# ============================================================
# 7. Rolling-window helper functions
# ============================================================

def resolve_ticker_universe(df, requested_tickers=None):
    available = sorted(df[TICKER_COL].dropna().astype(str).unique().tolist())
    if requested_tickers is None:
        return available
    requested = [str(t) for t in requested_tickers]
    missing = sorted(set(requested) - set(available))
    if missing:
        print('WARNING: requested tickers not found and will be skipped:', missing)
    return [t for t in requested if t in set(available)]


def prepare_xy(df, feature_cols, target_col, task):
    data = df[[c for c in feature_cols if c in df.columns] + [target_col]].copy()
    data = data.dropna(subset=[target_col])
    X = data[[c for c in feature_cols if c in data.columns]].apply(pd.to_numeric, errors='coerce')
    y = data[target_col]
    if task == 'regression':
        y = pd.to_numeric(y, errors='coerce')
    valid = y.notna()
    return X.loc[valid], y.loc[valid]


def is_rl_target(target_name):
    return str(target_name).startswith('rl_')


def filter_rows_for_target(df, target_name):
    df = df.copy()
    if is_rl_target(target_name) and RL_TRAINABLE_COL in df.columns:
        return df[pd.to_numeric(df[RL_TRAINABLE_COL], errors='coerce') == 1].copy()
    return df


def ensure_year_column(df):
    df = df.copy()
    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
    if YEAR_COL not in df.columns or df[YEAR_COL].isna().all():
        if DATE_COL not in df.columns:
            raise ValueError(f'Missing both {YEAR_COL} and {DATE_COL}.')
        df[YEAR_COL] = df[DATE_COL].dt.year
    df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors='coerce').astype('Int64')
    return df


def build_single_ticker_rolling_folds(df, ticker, rolling_train_years=3, n_test_folds=3, test_years=None):
    """Build rolling-window folds for one ticker.

    For each test year Y, the training window contains the most recent
    `rolling_train_years` years before Y.

    Example with rolling_train_years=3:
        2021-2023 -> 2024
        2022-2024 -> 2025
        2023-2025 -> 2026
    """
    one = df[df[TICKER_COL].astype(str) == str(ticker)].copy()
    years = sorted(one[YEAR_COL].dropna().astype(int).unique().tolist())
    year_set = set(years)

    def train_years_for(test_year):
        # Require the exact recent rolling window to exist.
        return [int(y) for y in range(int(test_year) - int(rolling_train_years), int(test_year))]

    if test_years is None:
        feasible = []
        for y in years:
            train_years = train_years_for(y)
            if len(train_years) == rolling_train_years and all(ty in year_set for ty in train_years):
                feasible.append(y)
        test_years = feasible[-n_test_folds:]
    else:
        test_years = [int(y) for y in test_years]

    folds = []
    for test_year in test_years:
        train_years = train_years_for(test_year)
        if len(train_years) != rolling_train_years or not all(ty in year_set for ty in train_years):
            continue
        folds.append({
            'ticker': ticker,
            'walkforward_scheme': f'rolling_{rolling_train_years}y',
            'window_label': f'{min(train_years)}-{max(train_years)}_to_{test_year}',
            'fold_id': f'{ticker}_rolling{rolling_train_years}y_{min(train_years)}_{max(train_years)}_test_{test_year}',
            'train_years': train_years,
            'train_start_year': min(train_years),
            'train_end_year': max(train_years),
            'test_year': test_year,
        })
    return folds

def model_classes(model):
    if hasattr(model, 'classes_'):
        return list(model.classes_)
    if hasattr(model, 'named_steps'):
        last = list(model.named_steps.values())[-1]
        if hasattr(last, 'classes_'):
            return list(last.classes_)
    return None


def score_for_ranking(model, X, y_pred, task, target_name):
    if task == 'regression':
        return np.asarray(y_pred, dtype=float)
    if not hasattr(model, 'predict_proba'):
        return np.asarray(y_pred)
    proba = model.predict_proba(X)
    classes = model_classes(model)
    if classes is None:
        return proba[:, -1] if proba.ndim == 2 else proba
    classes_str = [str(c).lower() for c in classes]
    if 'long' in classes_str:
        return proba[:, classes_str.index('long')]
    if '1' in classes_str:
        return proba[:, classes_str.index('1')]
    if 1 in classes:
        return proba[:, classes.index(1)]
    numeric = pd.to_numeric(pd.Series(classes), errors='coerce')
    if numeric.notna().all():
        return np.dot(proba, numeric.to_numpy(dtype=float))
    return np.max(proba, axis=1)


def actual_value_for_ranking(y, target_name):
    s = pd.Series(y).copy()
    if target_name == 'rl_expert_action':
        return s.astype(str).str.lower().eq('long').astype(float)
    return pd.to_numeric(s, errors='coerce').astype(float)


def percentile_against_train_distribution(test_scores, train_scores):
    train_scores = pd.to_numeric(pd.Series(train_scores), errors='coerce').dropna().sort_values().to_numpy()
    test_scores = pd.to_numeric(pd.Series(test_scores), errors='coerce').to_numpy()
    if len(train_scores) == 0:
        return pd.Series(np.nan, index=range(len(test_scores)))
    pct = np.searchsorted(train_scores, test_scores, side='right') / len(train_scores)
    return pd.Series(np.clip(pct, 0, 1))


def top_bottom_diagnostics(y_true, score, pct=0.10, target_name=None):
    y_rank = actual_value_for_ranking(y_true, target_name)
    score = pd.to_numeric(pd.Series(score), errors='coerce')
    d = pd.DataFrame({'y_rank': y_rank, 'score': score}).dropna()
    if len(d) < 10 or d['score'].nunique() < 2:
        return {'top_n': 0, 'bottom_n': 0, 'top_mean_true_rank_value': np.nan, 'bottom_mean_true_rank_value': np.nan, 'top_minus_bottom_mean_true': np.nan, 'top_positive_rate': np.nan, 'bottom_positive_rate': np.nan, 'top_bottom_lift': np.nan}
    n = max(1, int(np.ceil(len(d) * pct)))
    ranked = d.sort_values('score', ascending=False)
    top = ranked.head(n)
    bottom = ranked.tail(n)
    top_pos = (top['y_rank'] > 0).mean()
    bottom_pos = (bottom['y_rank'] > 0).mean()
    return {
        'top_n': int(len(top)),
        'bottom_n': int(len(bottom)),
        'top_mean_score': top['score'].mean(),
        'bottom_mean_score': bottom['score'].mean(),
        'top_mean_true_rank_value': top['y_rank'].mean(),
        'bottom_mean_true_rank_value': bottom['y_rank'].mean(),
        'top_minus_bottom_mean_true': top['y_rank'].mean() - bottom['y_rank'].mean(),
        'top_positive_rate': top_pos,
        'bottom_positive_rate': bottom_pos,
        'top_bottom_lift': top_pos / bottom_pos if bottom_pos and bottom_pos > 0 else np.nan,
    }


In [8]:
# ============================================================
# 8. Fit one rolling fold and calculate metrics
# ============================================================
def prediction_frame(test_df, y_test, y_pred, score, signal_score, fold, target_name, task, feature_set, model_name):
    meta_cols = [INDEX_COL, DATE_COL, YEAR_COL, TICKER_COL, SIC2_COL, SPLIT_COL]
    meta = test_df.loc[y_test.index, [c for c in meta_cols if c in test_df.columns]].copy()
    out = meta.copy()
    out['ticker'] = fold['ticker']
    out['target_name'] = target_name
    out['task'] = task
    out['feature_set'] = feature_set
    out['model_name'] = model_name
    out['model'] = model_name
    out['walkforward_scheme'] = fold['walkforward_scheme']
    out['window_label'] = fold['window_label']
    out['fold_id'] = fold['fold_id']
    out['train_start_year'] = fold['train_start_year']
    out['train_end_year'] = fold['train_end_year']
    out['test_year'] = fold['test_year']
    out[TRUE_COL] = np.asarray(y_test)
    out[PRED_COL] = np.asarray(y_pred)
    out[SCORE_COL] = np.asarray(score)
    out[SIGNAL_SCORE_COL] = np.asarray(signal_score)
    # Neutral placeholder. The row-level outputs do not hard-code long/no_trade direction.
    out[CONFIDENCE_COL] = 1.0
    out['actual_value_for_ranking'] = actual_value_for_ranking(out[TRUE_COL], target_name).to_numpy()
    return out


def metrics_for_prediction_frame(pred, target_name, task):
    out = {'n': len(pred), 'target_name': target_name, 'task': task}
    y_true = pred[TRUE_COL]
    y_pred = pred[PRED_COL]
    score = pred[SCORE_COL]
    y_rank = pred['actual_value_for_ranking']
    if task == 'regression':
        out.update(regression_metrics(y_true, y_pred))
    elif task == 'binary':
        out.update(binary_metrics(y_true, y_pred, score))
    elif task == 'multiclass':
        out.update(multiclass_metrics(y_true, y_pred))
    out['within_ticker_spearman'] = safe_spearman(y_rank, score)
    out['within_ticker_pearson'] = safe_pearson(y_rank, score)
    out.update(top_bottom_diagnostics(y_true, score, TOP_BOTTOM_PCT, target_name))
    return out


def fit_predict_one_fold(base, fold, target_name, target_cfg, feature_cols, model_name, model):
    target_col = target_cfg['column']
    task = target_cfg['task']
    ticker_df = base[base[TICKER_COL].astype(str) == str(fold['ticker'])].copy()
    train_df = ticker_df[ticker_df[YEAR_COL].astype(int).isin(fold['train_years'])].copy()
    test_df = ticker_df[ticker_df[YEAR_COL].astype(int) == int(fold['test_year'])].copy()
    train_df = filter_rows_for_target(train_df, target_name)
    test_df = filter_rows_for_target(test_df, target_name)
    X_train, y_train = prepare_xy(train_df, feature_cols, target_col, task)
    X_test, y_test = prepare_xy(test_df, feature_cols, target_col, task)
    if len(y_train) < MIN_TRAIN_ROWS or len(y_test) < MIN_TEST_ROWS:
        return None, None, {'status': 'skipped', 'reason': 'too_few_rows', 'n_train': len(y_train), 'n_test': len(y_test), **fold}
    if task in ['binary', 'multiclass'] and y_train.nunique() < 2:
        return None, None, {'status': 'skipped', 'reason': 'only_one_train_class', 'n_train': len(y_train), 'n_test': len(y_test), **fold}

    fitted = clone(model)
    fitted.fit(X_train, y_train)
    y_pred = fitted.predict(X_test)
    test_score = score_for_ranking(fitted, X_test, y_pred, task, target_name)
    train_pred = fitted.predict(X_train)
    train_score = score_for_ranking(fitted, X_train, train_pred, task, target_name)
    signal_score = percentile_against_train_distribution(test_score, train_score)

    pred = prediction_frame(test_df, y_test, y_pred, test_score, signal_score, fold, target_name, task, FEATURE_SET_NAME, model_name)
    metrics = metrics_for_prediction_frame(pred, target_name, task)
    metrics.update({
        'ticker': fold['ticker'],
        'feature_set': FEATURE_SET_NAME,
        'model': model_name,
        'walkforward_scheme': fold['walkforward_scheme'],
        'window_label': fold['window_label'],
        'fold_id': fold['fold_id'],
        'train_start_year': fold['train_start_year'],
        'train_end_year': fold['train_end_year'],
        'test_year': fold['test_year'],
        'n_train': len(y_train),
        'n_test': len(y_test),
    })
    return metrics, pred, {'status': 'completed', 'n_train': len(y_train), 'n_test': len(y_test), **fold}


In [9]:
# ============================================================
# 9. Main each-ticker rolling-window runner
# ============================================================
def run_each_ticker_rolling(base, feature_sets, requested_tickers=None):
    base = ensure_year_column(base)
    ticker_universe = resolve_ticker_universe(base, requested_tickers)
    print('Ticker universe size:', len(ticker_universe))

    available_targets = {
        k: v for k, v in TARGET_CONFIGS.items()
        if v['column'] in base.columns and base[v['column']].notna().any()
    }
    print('Available targets:', list(available_targets.keys()))

    if FEATURE_SET_NAME not in feature_sets:
        raise ValueError(f'Missing feature set {FEATURE_SET_NAME}. Available: {list(feature_sets)}')
    feature_cols = [c for c in feature_sets[FEATURE_SET_NAME] if c in base.columns]
    if not feature_cols:
        raise ValueError('No feature columns available for selected feature set.')
    print('Feature set:', FEATURE_SET_NAME, '| n_features:', len(feature_cols))

    model_name_by_task = {
        'regression': REGRESSION_MODELS,
        'binary': BINARY_MODELS,
        'multiclass': MULTICLASS_MODELS,
    }

    fold_rows, metric_rows, pred_parts, skipped_rows = [], [], [], []

    for ticker in ticker_universe:
        folds = build_single_ticker_rolling_folds(
            base, ticker, ROLLING_TRAIN_YEARS, N_TEST_FOLDS, TEST_YEARS
        )
        fold_rows.extend(folds)

        if not folds:
            skipped_rows.append({
                'ticker': ticker, 'status': 'skipped', 'reason': 'no_feasible_folds'
            })
            continue

        for target_name in TARGET_NAMES:
            if target_name not in available_targets:
                skipped_rows.append({
                    'ticker': ticker, 'target_name': target_name,
                    'status': 'skipped', 'reason': 'target_unavailable'
                })
                continue

            target_cfg = available_targets[target_name]
            models = get_models_for_task(target_cfg['task'])

            for model_name in model_name_by_task[target_cfg['task']]:
                if model_name not in models:
                    skipped_rows.append({
                        'ticker': ticker, 'target_name': target_name,
                        'model': model_name, 'status': 'skipped',
                        'reason': 'model_unavailable'
                    })
                    continue

                for fold in folds:
                    print(
                        f"{ticker} | {target_name} | {model_name} | "
                        f"train {fold['train_start_year']}-{fold['train_end_year']} "
                        f"-> test {fold['test_year']}"
                    )
                    try:
                        metrics, pred, status = fit_predict_one_fold(
                            base, fold, target_name, target_cfg,
                            feature_cols, model_name, models[model_name]
                        )
                        if metrics is not None:
                            metrics['training_approach'] = 'single_ticker'
                            metric_rows.append(metrics)
                        if pred is not None and len(pred):
                            pred['training_approach'] = 'single_ticker'
                            pred_parts.append(pred)
                        if status.get('status') != 'completed':
                            status.update({'target_name': target_name, 'model': model_name})
                            skipped_rows.append(status)
                    except Exception as e:
                        skipped_rows.append({
                            'ticker': ticker, 'target_name': target_name,
                            'model': model_name, 'fold_id': fold['fold_id'],
                            'status': 'error', 'reason': str(e)
                        })
                        print('  ERROR:', e)

    folds_df = pd.DataFrame(fold_rows)
    metrics_by_fold = pd.DataFrame(metric_rows)
    predictions = (
        pd.concat(pred_parts, ignore_index=True, sort=False)
        if pred_parts else pd.DataFrame()
    )
    skipped = pd.DataFrame(skipped_rows)

    # Overall result for each ticker, combining all its OOS test rows.
    ticker_overall_rows = []
    if len(predictions):
        group_cols = [
            'ticker', 'target_name', 'task', 'feature_set',
            'model', 'walkforward_scheme', 'training_approach'
        ]
        for keys, g in predictions.groupby(group_cols, dropna=False):
            row = metrics_for_prediction_frame(g, keys[1], keys[2])
            for col, value in zip(group_cols, keys):
                row[col] = value
            row['n_folds'] = g['fold_id'].nunique()
            row['test_years'] = ','.join(
                map(str, sorted(g['test_year'].dropna().astype(int).unique()))
            )
            ticker_overall_rows.append(row)

    metrics_ticker_overall = pd.DataFrame(ticker_overall_rows)
    return (
        metrics_by_fold, metrics_ticker_overall,
        predictions, skipped, folds_df, ticker_universe
    )

In [10]:
# ============================================================
# 10. Load data and run each-ticker rolling experiment
# ============================================================
frames, all_data = load_dataset_from_config(DATASET_CONFIG)
frames = {split: add_derived_targets(df) for split, df in frames.items()}
all_data = pd.concat(frames.values(), ignore_index=True, sort=False)
all_data = ensure_year_column(all_data)

FEATURE_SETS = build_feature_sets(all_data, DATASET_CONFIG)

(
    metrics_by_fold,
    metrics_ticker_overall,
    row_level_predictions,
    skipped,
    folds,
    ticker_universe,
) = run_each_ticker_rolling(all_data, FEATURE_SETS, TICKERS)

print('ticker_universe:', len(ticker_universe))
print('metrics_by_fold:', metrics_by_fold.shape)
print('metrics_ticker_overall:', metrics_ticker_overall.shape)
print('row_level_predictions:', row_level_predictions.shape)
print('skipped:', skipped.shape)
print('folds:', folds.shape)

train (98419, 750) universe_100_recent_post_normalisation_train.jsonl
valid (24469, 750) universe_100_recent_post_normalisation_validation.jsonl
test (32142, 750) universe_100_recent_post_normalisation_test.jsonl
Ticker universe size: 98
Available targets: ['pi_hindsight_entry_long', 'pi_hindsight_entry_positive', 'pi_hindsight_entry_original', 'pi_hindsight_entry_6bins', 'rl_expert_action', 'rl_long_action_binary', 'rl_long_action_quality', 'rl_long_current_pnl']


ValueError: Missing feature set full_feature_B. Available: ['combined_project_b', 'all_numeric_features']

In [ ]:
# ============================================================
# 11. Aggregate within-ticker results and save outputs
# ============================================================
def summarise_ticker_distribution(metrics_ticker_overall):
    if metrics_ticker_overall.empty:
        return pd.DataFrame()

    group_cols = [
        'target_name', 'feature_set', 'model', 'walkforward_scheme', 'training_approach'
    ]
    rows = []

    for keys, g in metrics_ticker_overall.groupby(group_cols, dropna=False):
        s = pd.to_numeric(g['within_ticker_spearman'], errors='coerce').dropna()
        row = dict(zip(group_cols, keys))
        row.update({
            'n_tickers': int(g['ticker'].nunique()),
            'n_valid_tickers': int(len(s)),
            'mean_within_ticker_spearman': s.mean() if len(s) else np.nan,
            'median_within_ticker_spearman': s.median() if len(s) else np.nan,
            'std_within_ticker_spearman': s.std(ddof=1) if len(s) > 1 else np.nan,
            'variance_within_ticker_spearman': s.var(ddof=1) if len(s) > 1 else np.nan,
            'q10': s.quantile(0.10) if len(s) else np.nan,
            'q25': s.quantile(0.25) if len(s) else np.nan,
            'q75': s.quantile(0.75) if len(s) else np.nan,
            'q90': s.quantile(0.90) if len(s) else np.nan,
            'min': s.min() if len(s) else np.nan,
            'max': s.max() if len(s) else np.nan,
            'positive_ticker_rate': (s > 0).mean() if len(s) else np.nan,
        })
        rows.append(row)

    return pd.DataFrame(rows)


def summarise_fold_stability(metrics_by_fold):
    if metrics_by_fold.empty:
        return pd.DataFrame()

    group_cols = [
        'target_name', 'feature_set', 'model', 'walkforward_scheme',
        'training_approach', 'test_year'
    ]
    rows = []

    for keys, g in metrics_by_fold.groupby(group_cols, dropna=False):
        s = pd.to_numeric(g['within_ticker_spearman'], errors='coerce').dropna()
        row = dict(zip(group_cols, keys))
        row.update({
            'n_tickers': int(g['ticker'].nunique()),
            'n_valid_tickers': int(len(s)),
            'mean_within_ticker_spearman': s.mean() if len(s) else np.nan,
            'median_within_ticker_spearman': s.median() if len(s) else np.nan,
            'std_across_tickers': s.std(ddof=1) if len(s) > 1 else np.nan,
            'variance_across_tickers': s.var(ddof=1) if len(s) > 1 else np.nan,
            'positive_ticker_rate': (s > 0).mean() if len(s) else np.nan,
            'q25': s.quantile(0.25) if len(s) else np.nan,
            'q75': s.quantile(0.75) if len(s) else np.nan,
        })
        rows.append(row)

    return pd.DataFrame(rows)


def summarise_stability_across_folds(fold_stability):
    if fold_stability.empty:
        return pd.DataFrame()

    group_cols = [
        'target_name', 'feature_set', 'model', 'walkforward_scheme', 'training_approach'
    ]
    rows = []

    for keys, g in fold_stability.groupby(group_cols, dropna=False):
        fold_means = pd.to_numeric(
            g['mean_within_ticker_spearman'], errors='coerce'
        ).dropna()
        row = dict(zip(group_cols, keys))
        row.update({
            'n_test_folds': int(g['test_year'].nunique()),
            'mean_of_fold_means': fold_means.mean() if len(fold_means) else np.nan,
            'median_of_fold_means': fold_means.median() if len(fold_means) else np.nan,
            'std_of_fold_means': fold_means.std(ddof=1) if len(fold_means) > 1 else np.nan,
            'min_fold_mean': fold_means.min() if len(fold_means) else np.nan,
            'max_fold_mean': fold_means.max() if len(fold_means) else np.nan,
            'fold_range': (
                fold_means.max() - fold_means.min()
                if len(fold_means) else np.nan
            ),
            'positive_fold_rate': (fold_means > 0).mean() if len(fold_means) else np.nan,
        })
        rows.append(row)

    return pd.DataFrame(rows)


ticker_distribution = summarise_ticker_distribution(metrics_ticker_overall)
fold_stability = summarise_fold_stability(metrics_by_fold)
overall_fold_stability = summarise_stability_across_folds(fold_stability)

OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

output_files = {
    'metrics_by_fold': OUTPUT_BASE_DIR / 'single_ticker_metrics_by_fold.csv',
    'metrics_ticker_overall': OUTPUT_BASE_DIR / 'single_ticker_metrics_ticker_overall.csv',
    'ticker_distribution': OUTPUT_BASE_DIR / 'single_ticker_ticker_distribution.csv',
    'fold_stability': OUTPUT_BASE_DIR / 'single_ticker_fold_stability.csv',
    'overall_fold_stability': OUTPUT_BASE_DIR / 'single_ticker_overall_fold_stability.csv',
    'row_level_predictions_csv': OUTPUT_BASE_DIR / 'single_ticker_row_level_predictions.csv',
    'row_level_predictions_parquet': OUTPUT_BASE_DIR / 'single_ticker_row_level_predictions.parquet',
    'folds': OUTPUT_BASE_DIR / 'single_ticker_folds.csv',
    'skipped': OUTPUT_BASE_DIR / 'single_ticker_skipped.csv',
    'excel': OUTPUT_BASE_DIR / 'single_ticker_aggregated_results.xlsx',
}

metrics_by_fold.to_csv(output_files['metrics_by_fold'], index=False)
metrics_ticker_overall.to_csv(output_files['metrics_ticker_overall'], index=False)
ticker_distribution.to_csv(output_files['ticker_distribution'], index=False)
fold_stability.to_csv(output_files['fold_stability'], index=False)
overall_fold_stability.to_csv(output_files['overall_fold_stability'], index=False)
row_level_predictions.to_csv(output_files['row_level_predictions_csv'], index=False)
if not row_level_predictions.empty:
    row_level_predictions.to_parquet(
        output_files['row_level_predictions_parquet'], index=False
    )
folds.to_csv(output_files['folds'], index=False)
skipped.to_csv(output_files['skipped'], index=False)

with pd.ExcelWriter(output_files['excel'], engine='openpyxl') as writer:
    metrics_by_fold.to_excel(writer, sheet_name='ticker_fold_metrics', index=False)
    metrics_ticker_overall.to_excel(writer, sheet_name='ticker_overall', index=False)
    ticker_distribution.to_excel(writer, sheet_name='ticker_distribution', index=False)
    fold_stability.to_excel(writer, sheet_name='fold_stability', index=False)
    overall_fold_stability.to_excel(writer, sheet_name='overall_stability', index=False)
    folds.to_excel(writer, sheet_name='folds', index=False)
    skipped.to_excel(writer, sheet_name='skipped', index=False)
    row_level_predictions.head(50000).to_excel(
        writer, sheet_name='prediction_preview', index=False
    )

display(ticker_distribution)
display(fold_stability)
display(overall_fold_stability)

print('\nSaved files:')
for name, path in output_files.items():
    print(name, '->', path)

# Complete-year reporting sample for all headline single-ticker summaries.
metrics_by_fold_main_years = metrics_by_fold[
    metrics_by_fold['test_year'].isin(MAIN_TEST_YEARS)
].copy()

metrics_ticker_overall_main_years = (
    metrics_by_fold_main_years
    .groupby(['ticker', 'target_name', 'feature_set', 'model'], dropna=False)
    .agg(
        mean_fold_spearman=('within_ticker_spearman', 'mean'),
        median_fold_spearman=('within_ticker_spearman', 'median'),
        std_fold_spearman=('within_ticker_spearman', 'std'),
        min_fold_spearman=('within_ticker_spearman', 'min'),
        max_fold_spearman=('within_ticker_spearman', 'max'),
        positive_fold_rate=(
            'within_ticker_spearman',
            lambda s: (pd.to_numeric(s, errors='coerce') > 0).mean()
        ),
        n_folds=('test_year', 'nunique'),
    )
    .reset_index()
)

metrics_ticker_overall_main_years.to_csv(
    OUTPUT_BASE_DIR / 'single_ticker_overall_metrics_2022_2025.csv',
    index=False,
)

print('Single-ticker headline summaries use MAIN_TEST_YEARS:', MAIN_TEST_YEARS)
display(metrics_ticker_overall_main_years)


In [ ]:
# ============================================================
# 12. Exact pooled-versus-single comparison using IndexReference
# ============================================================
def load_prediction_table(path):
    path = Path(path)
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    raise ValueError('Use a .csv or .parquet prediction file.')


def normalise_prediction_columns(df, approach):
    df = df.copy()

    if 'ticker' not in df.columns and TICKER_COL in df.columns:
        df['ticker'] = df[TICKER_COL].astype(str)
    elif 'ticker' in df.columns:
        df['ticker'] = df['ticker'].astype(str)
    else:
        raise ValueError('Prediction table needs ticker or attr__ticker.')

    if 'test_year' not in df.columns:
        if YEAR_COL in df.columns:
            df['test_year'] = pd.to_numeric(df[YEAR_COL], errors='coerce')
        elif DATE_COL in df.columns:
            df['test_year'] = pd.to_datetime(
                df[DATE_COL], errors='coerce'
            ).dt.year
        else:
            raise ValueError('Prediction table needs test_year, year, or timestamp.')

    if 'model' not in df.columns and 'model_name' in df.columns:
        df['model'] = df['model_name']

    if 'IndexReference' not in df.columns:
        candidates = [
            'index_reference',
            'indexreference',
            'Index_Reference',
            'attr__IndexReference',
        ]
        found = next((c for c in candidates if c in df.columns), None)
        if found is None:
            raise ValueError(
                'Prediction table needs IndexReference for exact row matching.'
            )
        df['IndexReference'] = df[found]

    if 'feature_set' not in df.columns:
        if approach == 'single_ticker':
            df['feature_set'] = FEATURE_SET_NAME
        else:
            raise ValueError('Pooled prediction table needs feature_set.')

    if 'walkforward_scheme' not in df.columns:
        df['walkforward_scheme'] = np.nan

    required = [
        'IndexReference', 'ticker', 'test_year',
        'target_name', 'feature_set', 'model', TRUE_COL, SCORE_COL,
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing prediction columns: {missing}')

    if 'actual_value_for_ranking' not in df.columns:
        df['actual_value_for_ranking'] = pd.to_numeric(
            df[TRUE_COL], errors='coerce'
        )

    df['IndexReference'] = df['IndexReference'].astype(str)
    df['test_year'] = pd.to_numeric(df['test_year'], errors='coerce')
    df['training_approach'] = approach
    return df


def filter_comparable_predictions(df, approach):
    df = df.copy()

    if approach == 'pooled':
        df = df[
            df['walkforward_scheme'].astype(str).eq('rolling_2y')
        ].copy()

    df = df[df['target_name'].isin(TARGET_NAMES)].copy()
    df = df[df['feature_set'].astype(str).eq(FEATURE_SET_NAME)].copy()

    allowed_models = sorted(
        set(REGRESSION_MODELS + BINARY_MODELS + MULTICLASS_MODELS)
    )
    df = df[df['model'].isin(allowed_models)].copy()
    return df


def exact_match_prediction_rows(single_predictions, pooled_predictions):
    single = filter_comparable_predictions(
        normalise_prediction_columns(single_predictions, 'single_ticker'),
        'single_ticker',
    )
    pooled = filter_comparable_predictions(
        normalise_prediction_columns(pooled_predictions, 'pooled'),
        'pooled',
    )

    row_keys = [
        'IndexReference',
        'ticker',
        'test_year',
        'target_name',
        'feature_set',
        'model',
    ]

    if single.duplicated(row_keys).any():
        raise ValueError('Duplicate row keys found in single-ticker predictions.')
    if pooled.duplicated(row_keys).any():
        raise ValueError('Duplicate row keys found in pooled predictions.')

    single_cols = row_keys + [
        TRUE_COL,
        'actual_value_for_ranking',
        SCORE_COL,
        'fold_id',
        'walkforward_scheme',
    ]
    pooled_cols = row_keys + [
        TRUE_COL,
        'actual_value_for_ranking',
        SCORE_COL,
        'fold_id',
        'walkforward_scheme',
    ]
    single_cols = [c for c in single_cols if c in single.columns]
    pooled_cols = [c for c in pooled_cols if c in pooled.columns]

    matched = single[single_cols].merge(
        pooled[pooled_cols],
        on=row_keys,
        how='inner',
        suffixes=('_single', '_pooled'),
        validate='one_to_one',
    )

    if matched.empty:
        raise ValueError(
            'No rows matched exactly on IndexReference, ticker, test year, '
            'target, feature set and model.'
        )

    y_single_col = f'{TRUE_COL}_single'
    y_pooled_col = f'{TRUE_COL}_pooled'
    matched['same_y_true'] = np.isclose(
        pd.to_numeric(matched[y_single_col], errors='coerce'),
        pd.to_numeric(matched[y_pooled_col], errors='coerce'),
        equal_nan=True,
    )

    if not matched['same_y_true'].all():
        bad_n = int((~matched['same_y_true']).sum())
        raise ValueError(
            f'{bad_n} exactly matched rows have inconsistent y_true values.'
        )

    group_keys = ['ticker', 'test_year', 'target_name', 'feature_set', 'model']
    single_counts = (
        single.groupby(group_keys).size()
        .rename('single_rows').reset_index()
    )
    pooled_counts = (
        pooled.groupby(group_keys).size()
        .rename('pooled_rows').reset_index()
    )
    matched_counts = (
        matched.groupby(group_keys).size()
        .rename('matched_rows').reset_index()
    )

    coverage = (
        single_counts.merge(pooled_counts, on=group_keys, how='outer')
        .merge(matched_counts, on=group_keys, how='outer')
        .fillna({'single_rows': 0, 'pooled_rows': 0, 'matched_rows': 0})
    )
    coverage['single_match_rate'] = np.where(
        coverage['single_rows'] > 0,
        coverage['matched_rows'] / coverage['single_rows'],
        np.nan,
    )
    coverage['pooled_match_rate'] = np.where(
        coverage['pooled_rows'] > 0,
        coverage['matched_rows'] / coverage['pooled_rows'],
        np.nan,
    )
    coverage['complete_exact_match'] = (
        coverage['single_rows'].eq(coverage['matched_rows'])
        & coverage['pooled_rows'].eq(coverage['matched_rows'])
    )

    return matched, coverage


def calculate_ticker_level_pairs(matched):
    rows = []
    group_cols = ['ticker', 'target_name', 'feature_set', 'model']

    for keys, g in matched.groupby(group_cols, dropna=False):
        y = pd.to_numeric(g[f'{TRUE_COL}_single'], errors='coerce')
        single_score = pd.to_numeric(
            g[f'{SCORE_COL}_single'], errors='coerce'
        )
        pooled_score = pd.to_numeric(
            g[f'{SCORE_COL}_pooled'], errors='coerce'
        )

        row = dict(zip(group_cols, keys))
        row.update({
            'n_matched_rows': len(g),
            'n_common_test_folds': g['test_year'].nunique(),
            'common_test_years': ','.join(
                map(str, sorted(g['test_year'].dropna().astype(int).unique()))
            ),
            'single_spearman': safe_spearman(y, single_score),
            'pooled_spearman': safe_spearman(y, pooled_score),
        })
        row['difference_single_minus_pooled'] = (
            row['single_spearman'] - row['pooled_spearman']
        )
        rows.append(row)

    return pd.DataFrame(rows)


def summarise_ticker_level_pairs(paired_ticker):
    rows = []
    group_cols = ['target_name', 'feature_set', 'model']

    for keys, g in paired_ticker.groupby(group_cols, dropna=False):
        g = g.dropna(
            subset=['single_spearman', 'pooled_spearman']
        ).copy()

        single_s = g['single_spearman'].astype(float)
        pooled_s = g['pooled_spearman'].astype(float)
        diff = g['difference_single_minus_pooled'].astype(float)

        if len(g) >= 2 and diff.std(ddof=1) > 0:
            t_stat, t_p = stats.ttest_rel(
                single_s, pooled_s, nan_policy='omit'
            )
        else:
            t_stat, t_p = np.nan, np.nan

        try:
            w_stat, w_p = stats.wilcoxon(
                single_s,
                pooled_s,
                zero_method='wilcox',
                alternative='two-sided',
            )
        except ValueError:
            w_stat, w_p = np.nan, np.nan

        single_var = single_s.var(ddof=1) if len(single_s) > 1 else np.nan
        pooled_var = pooled_s.var(ddof=1) if len(pooled_s) > 1 else np.nan

        row = dict(zip(group_cols, keys))
        row.update({
            'n_ticker_pairs': len(g),
            'single_mean_spearman': single_s.mean(),
            'single_median_spearman': single_s.median(),
            'pooled_mean_spearman': pooled_s.mean(),
            'pooled_median_spearman': pooled_s.median(),
            'mean_paired_difference': diff.mean(),
            'median_paired_difference': diff.median(),
            'single_win_rate': (diff > 0).mean(),
            'single_positive_ticker_rate': (single_s > 0).mean(),
            'pooled_positive_ticker_rate': (pooled_s > 0).mean(),
            'paired_t_stat': t_stat,
            'paired_t_pvalue': t_p,
            'wilcoxon_stat': w_stat,
            'wilcoxon_pvalue': w_p,
            'cohen_dz': (
                diff.mean() / diff.std(ddof=1)
                if len(diff) > 1 and diff.std(ddof=1) > 0
                else np.nan
            ),
            'single_std': single_s.std(ddof=1),
            'pooled_std': pooled_s.std(ddof=1),
            'single_variance': single_var,
            'pooled_variance': pooled_var,
            'variance_ratio_single_over_pooled': (
                single_var / pooled_var
                if pd.notna(single_var)
                and pd.notna(pooled_var)
                and pooled_var > 0
                else np.nan
            ),
            'difference_std': diff.std(ddof=1),
            'difference_q25': diff.quantile(0.25),
            'difference_q75': diff.quantile(0.75),
        })
        rows.append(row)

    return pd.DataFrame(rows)


def calculate_fold_level_pairs(matched):
    rows = []
    group_cols = ['ticker', 'test_year', 'target_name', 'feature_set', 'model']

    for keys, g in matched.groupby(group_cols, dropna=False):
        y = pd.to_numeric(g[f'{TRUE_COL}_single'], errors='coerce')
        single_score = pd.to_numeric(
            g[f'{SCORE_COL}_single'], errors='coerce'
        )
        pooled_score = pd.to_numeric(
            g[f'{SCORE_COL}_pooled'], errors='coerce'
        )

        row = dict(zip(group_cols, keys))
        row.update({
            'n_matched_rows': len(g),
            'single_spearman': safe_spearman(y, single_score),
            'pooled_spearman': safe_spearman(y, pooled_score),
        })
        row['difference_single_minus_pooled'] = (
            row['single_spearman'] - row['pooled_spearman']
        )
        rows.append(row)

    return pd.DataFrame(rows)


def summarise_fold_stability(paired_fold):
    return (
        paired_fold
        .groupby(['target_name', 'feature_set', 'model', 'test_year'], dropna=False)
        .agg(
            n_ticker_pairs=('ticker', 'size'),
            single_mean_spearman=('single_spearman', 'mean'),
            single_median_spearman=('single_spearman', 'median'),
            pooled_mean_spearman=('pooled_spearman', 'mean'),
            pooled_median_spearman=('pooled_spearman', 'median'),
            mean_difference=('difference_single_minus_pooled', 'mean'),
            median_difference=('difference_single_minus_pooled', 'median'),
            single_win_rate=(
                'difference_single_minus_pooled',
                lambda s: (s > 0).mean(),
            ),
            single_std_across_tickers=('single_spearman', 'std'),
            pooled_std_across_tickers=('pooled_spearman', 'std'),
            difference_std=('difference_single_minus_pooled', 'std'),
        )
        .reset_index()
    )


if POOLED_PREDICTIONS_PATH is None:
    print('Set POOLED_PREDICTIONS_PATH to the pooled parquet file.')
else:
    pooled_row_level_predictions = load_prediction_table(
        POOLED_PREDICTIONS_PATH
    )

    exact_matched_predictions, exact_match_coverage_audit = (
        exact_match_prediction_rows(
            row_level_predictions,
            pooled_row_level_predictions,
        )
    )

    # ========================================================
    # Main complete-year sample: 2022–2025 only
    # ========================================================
    main_matched_predictions = exact_matched_predictions[
        exact_matched_predictions['test_year'].isin(MAIN_TEST_YEARS)
    ].copy()

    if main_matched_predictions.empty:
        raise ValueError(
            f'No exact-matched observations were found for MAIN_TEST_YEARS='
            f'{MAIN_TEST_YEARS}.'
        )

    primary_paired_ticker = calculate_ticker_level_pairs(
        main_matched_predictions
    )
    primary_paired_summary = summarise_ticker_level_pairs(
        primary_paired_ticker
    )

    main_paired_ticker_fold = calculate_fold_level_pairs(
        main_matched_predictions
    )
    main_paired_fold_stability = summarise_fold_stability(
        main_paired_ticker_fold
    )

    # ========================================================
    # Partial-year sample: 2026 only, descriptive/robustness
    # ========================================================
    partial_matched_predictions = exact_matched_predictions[
        exact_matched_predictions['test_year'].isin(PARTIAL_TEST_YEARS)
    ].copy()

    if not partial_matched_predictions.empty:
        partial_paired_ticker_fold = calculate_fold_level_pairs(
            partial_matched_predictions
        )
        partial_fold_summary = summarise_fold_stability(
            partial_paired_ticker_fold
        )
    else:
        partial_paired_ticker_fold = pd.DataFrame()
        partial_fold_summary = pd.DataFrame()

    # Save exact matched rows and all comparison tables.
    exact_matched_predictions.to_parquet(
        OUTPUT_BASE_DIR / 'exact_matched_pooled_single_predictions_all_years.parquet',
        index=False,
    )
    main_matched_predictions.to_parquet(
        OUTPUT_BASE_DIR / 'exact_matched_pooled_single_predictions_2022_2025.parquet',
        index=False,
    )
    exact_match_coverage_audit.to_csv(
        OUTPUT_BASE_DIR / 'exact_match_coverage_audit_all_years.csv',
        index=False,
    )

    # Main outputs: all averages/statistical comparisons are 2022–2025.
    primary_paired_ticker.to_csv(
        OUTPUT_BASE_DIR / 'primary_paired_ticker_comparison_2022_2025.csv',
        index=False,
    )
    primary_paired_summary.to_csv(
        OUTPUT_BASE_DIR / 'primary_paired_ticker_summary_2022_2025.csv',
        index=False,
    )
    main_paired_ticker_fold.to_csv(
        OUTPUT_BASE_DIR / 'main_paired_ticker_fold_comparison_2022_2025.csv',
        index=False,
    )
    main_paired_fold_stability.to_csv(
        OUTPUT_BASE_DIR / 'main_paired_fold_stability_2022_2025.csv',
        index=False,
    )

    # Partial 2026 outputs are kept separate and excluded from main averages.
    if not partial_paired_ticker_fold.empty:
        partial_matched_predictions.to_parquet(
            OUTPUT_BASE_DIR / 'partial_2026_exact_matched_predictions.parquet',
            index=False,
        )
        partial_paired_ticker_fold.to_csv(
            OUTPUT_BASE_DIR / 'partial_2026_paired_ticker_fold_comparison.csv',
            index=False,
        )
        partial_fold_summary.to_csv(
            OUTPUT_BASE_DIR / 'partial_2026_fold_summary.csv',
            index=False,
        )

    print('Exact matched rows, all years:', len(exact_matched_predictions))
    print('Main matched rows, 2022–2025:', len(main_matched_predictions))
    print('Partial matched rows, 2026:', len(partial_matched_predictions))
    print(
        'Complete exact-match groups:',
        int(exact_match_coverage_audit['complete_exact_match'].sum()),
        '/',
        len(exact_match_coverage_audit),
    )

    print('\nPRIMARY: one pair per ticker using complete years 2022–2025')
    display(primary_paired_summary)
    display(
        primary_paired_ticker.sort_values(
            'difference_single_minus_pooled',
            ascending=False,
        )
    )

    print('\nMAIN FOLD-LEVEL STABILITY: 2022–2025 only')
    display(main_paired_fold_stability)

    print('\nPARTIAL-YEAR RESULT: 2026 only, excluded from all main averages')
    if partial_fold_summary.empty:
        print('No matched 2026 observations were available.')
    else:
        display(partial_fold_summary)

    print('\nEXACT-MATCH COVERAGE AUDIT: all available years')
    display(exact_match_coverage_audit)
